# Формирование manual-check выборки из GPT-5-размеченного train set

Этот ноутбук собирает случайное подмножество примеров для ручной проверки качества разметки.

## Зачем нужен ноутбук

На предыдущем этапе весь train set был размечен через GPT-5:

```text
data/labeled/wb_feedbacks_ChatGpt_markup_from_synthetic_gpt5_V_2/
chatgpt_labeled_reviews_mvp_for_training.csv
```

После этого из GPT-5-размеченного train set выбирается случайное подмножество примеров по классам.  
Эта выборка нужна не для обучения модели, а для ручной проверки качества разметки. Чтобы подобрать итоговый промпт для разметки train данных. 

Данный набор стал golden set

## Что делает ноутбук

1. Загружает GPT-5-размеченный train set.
2. Для каждого класса выбирает случайные примеры.
3. Собирает единый файл для ручной проверки.
4. Добавляет поля для ручной валидации: `correct_labels`, `comment`, `is_correct`.
5. Сохраняет manual-check файл.

## Выходные данные

```text
data/labeled/wb_feedbacks_manual_check_random_gpt5/
manual_check_random_by_class.csv
```

После ручного заполнения `correct_labels` этот файл используется как golden set для оценки качества классификатора.

## Роль в пайплайне

```text
GPT-5-размеченный train set
        ↓
generate_manual_check_random_by_class_gpt5.ipynb
        ↓
случайное подмножество для ручной проверки
        ↓
ручная разметка correct_labels
        ↓
golden set для оценки модели
```

Важно: сначала через GPT-5 была размечена вся обучающая корзинка, и только потом из нее было выбрано случайное подмножество для ручной проверки.

In [1]:
# В Google Colab раскомментируй:
from google.colab import drive
drive.mount("/content/drive")

import json
import re
from pathlib import Path
from typing import Any

import pandas as pd


Mounted at /content/drive


## 1. Настройки

`PROJECT_ROOT` должен указывать на папку `data`, внутри которой лежит папка `labeled`.

Пример структуры:

```text
PROJECT_ROOT/
    labeled/
        wb_feedbacks_by_class_with_synthetic/
            проблема_с_качеством_товара.csv
            ...
```


In [2]:
# ====== ПУТИ ======

PROJECT_ROOT = Path("/content/drive/MyDrive/MLops_project/project/data").resolve()
LABELED_DIR = PROJECT_ROOT / "labeled"

INPUT_BY_CLASS_DIR = LABELED_DIR / "wb_feedbacks_by_class_with_synthetic"

OUTPUT_DIR = LABELED_DIR / "wb_feedbacks_manual_check_random"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "manual_check_random_by_class.csv"
SUMMARY_OUTPUT_PATH = OUTPUT_DIR / "manual_check_random_summary.csv"

# ====== НАСТРОЙКИ ВЫБОРКИ ======

N_REAL_PER_CLASS = 10
N_SYNTHETIC_PER_CLASS = 10

RANDOM_STATE = 42

TEXT_COLUMN = "отзыв"
LABELS_COLUMN = "labels"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("INPUT_BY_CLASS_DIR:", INPUT_BY_CLASS_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


PROJECT_ROOT: /content/drive/MyDrive/MLops_project/project/data
INPUT_BY_CLASS_DIR: /content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_by_class_with_synthetic
OUTPUT_DIR: /content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_manual_check_random


In [3]:
ALLOWED_LABELS_MVP = [
    "Проблема с качеством товара",
    "Проблема с размером / посадкой",
    "Несоответствие карточке товара",
    "Проблема с комплектацией / упаковкой",
    "Проблема доставки / получения",
    "Цена / ценность",
    "Проблема с возвратом",
    "Положительный / нейтральный отзыв",
    "Другая проблема",
]

ALLOWED_LABELS_SET = set(ALLOWED_LABELS_MVP)


## 2. Вспомогательные функции

In [4]:
def read_csv_safely(path: Path) -> pd.DataFrame:
    encodings = ["utf-8", "utf-8-sig", "cp1251"]
    last_error = None

    for encoding in encodings:
        try:
            return pd.read_csv(path, encoding=encoding)
        except Exception as e:
            last_error = e

    raise RuntimeError(f"Не удалось прочитать файл {path}. Последняя ошибка: {last_error}")


def labels_to_list(labels: Any) -> list[str]:
    if isinstance(labels, list):
        raw = labels
    elif pd.isna(labels):
        raw = []
    elif isinstance(labels, str):
        value = labels.strip()

        if not value:
            raw = []
        else:
            try:
                parsed = json.loads(value)
                raw = parsed if isinstance(parsed, list) else []
            except Exception:
                if " | " in value:
                    raw = [x.strip() for x in value.split(" | ")]
                elif value in ALLOWED_LABELS_SET:
                    raw = [value]
                else:
                    raw = []
    else:
        raw = []

    result = []
    for label in raw:
        label = str(label).strip()
        if label in ALLOWED_LABELS_SET and label not in result:
            result.append(label)

    return result


def normalize_labels(labels: Any) -> list[str]:
    result = labels_to_list(labels)

    if not result:
        result = ["Другая проблема"]

    neutral_label = "Положительный / нейтральный отзыв"

    if neutral_label in result and len(result) > 1:
        result = [label for label in result if label != neutral_label]

    if not result:
        result = ["Другая проблема"]

    return result


def safe_filename(label: str) -> str:
    name = label.lower()
    name = name.replace(" / ", "_")
    name = name.replace("/", "_")
    name = name.replace(" ", "_")
    name = re.sub(r"[^а-яa-z0-9_]+", "", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name + ".csv"


def sample_or_all(df: pd.DataFrame, n: int, random_state: int) -> pd.DataFrame:
    if df.empty:
        return df.copy()

    return df.sample(
        n=min(n, len(df)),
        random_state=random_state,
    ).copy()


## 3. Проверка входной папки и файлов по классам

In [5]:
if not INPUT_BY_CLASS_DIR.exists():
    raise FileNotFoundError(
        f"Папка с файлами по классам не найдена: {INPUT_BY_CLASS_DIR}"
    )

expected_files = []

for label in ALLOWED_LABELS_MVP:
    path = INPUT_BY_CLASS_DIR / safe_filename(label)
    expected_files.append({
        "label": label,
        "file": path,
        "exists": path.exists(),
    })

files_df = pd.DataFrame(expected_files)
files_df


,label,file,exists
0,Проблема с качеством товара,/content/drive/MyDrive/MLops_project/project/d...,True
1,Проблема с размером / посадкой,/content/drive/MyDrive/MLops_project/project/d...,True
2,Несоответствие карточке товара,/content/drive/MyDrive/MLops_project/project/d...,True
3,Проблема с комплектацией / упаковкой,/content/drive/MyDrive/MLops_project/project/d...,True
4,Проблема доставки / получения,/content/drive/MyDrive/MLops_project/project/d...,True
5,Цена / ценность,/content/drive/MyDrive/MLops_project/project/d...,True
6,Проблема с возвратом,/content/drive/MyDrive/MLops_project/project/d...,True
7,Положительный / нейтральный отзыв,/content/drive/MyDrive/MLops_project/project/d...,True
8,Другая проблема,/content/drive/MyDrive/MLops_project/project/d...,True


## 4. Формирование случайной выборки

Для каждого класса читается свой файл.  
Из него отдельно берутся реальные и синтетические примеры по колонке `source_type`.

Если `source_type` отсутствует, все строки считаются `real`.


In [6]:
sample_parts = []
summary_rows = []

for label in ALLOWED_LABELS_MVP:
    class_path = INPUT_BY_CLASS_DIR / safe_filename(label)

    if not class_path.exists():
        print("Файл не найден, пропускаю:", class_path)
        summary_rows.append({
            "class_for_check": label,
            "file": str(class_path),
            "real_available": 0,
            "synthetic_available": 0,
            "real_sampled": 0,
            "synthetic_sampled": 0,
            "total_sampled": 0,
            "status": "file_not_found",
        })
        continue

    part = read_csv_safely(class_path)

    if TEXT_COLUMN not in part.columns:
        raise ValueError(
            f"В файле {class_path} нет колонки {TEXT_COLUMN!r}. "
            f"Доступные колонки: {list(part.columns)}"
        )

    if LABELS_COLUMN not in part.columns:
        raise ValueError(
            f"В файле {class_path} нет колонки {LABELS_COLUMN!r}. "
            f"Доступные колонки: {list(part.columns)}"
        )

    part = part.copy()
    part[TEXT_COLUMN] = part[TEXT_COLUMN].fillna("").astype(str).str.strip()
    part = part[part[TEXT_COLUMN].ne("")].copy()

    part["labels_list"] = part[LABELS_COLUMN].apply(normalize_labels)
    part["labels_str"] = part["labels_list"].apply(lambda x: " | ".join(x))

    if "source_type" not in part.columns:
        part["source_type"] = "real"

    part["source_type"] = (
        part["source_type"]
        .fillna("real")
        .astype(str)
        .str.lower()
        .str.strip()
    )

    # Нормализуем возможные варианты названий.
    part.loc[part["source_type"].isin(["synth", "synthetic", "generated"]), "source_type"] = "synthetic"
    part.loc[~part["source_type"].isin(["real", "synthetic"]), "source_type"] = "real"

    # Оставляем только строки, где целевой класс реально есть в labels.
    # Это защита от случайных ошибок в файлах.
    part = part[part["labels_list"].apply(lambda labels: label in labels)].copy()

    real_part = part[part["source_type"] == "real"].copy()
    synthetic_part = part[part["source_type"] == "synthetic"].copy()

    real_sample = sample_or_all(real_part, N_REAL_PER_CLASS, RANDOM_STATE)
    synthetic_sample = sample_or_all(synthetic_part, N_SYNTHETIC_PER_CLASS, RANDOM_STATE)

    class_sample = pd.concat([real_sample, synthetic_sample], ignore_index=True)

    if not class_sample.empty:
        class_sample["class_for_check"] = label
        class_sample["source_class_file"] = str(class_path.relative_to(PROJECT_ROOT))
        class_sample["manual_check_id"] = [
            f"{safe_filename(label).replace('.csv', '')}__{i}"
            for i in range(len(class_sample))
        ]

        sample_parts.append(class_sample)

    summary_rows.append({
        "class_for_check": label,
        "file": str(class_path.relative_to(PROJECT_ROOT)),
        "real_available": len(real_part),
        "synthetic_available": len(synthetic_part),
        "real_sampled": len(real_sample),
        "synthetic_sampled": len(synthetic_sample),
        "total_sampled": len(class_sample),
        "status": "ok",
    })

summary_df = pd.DataFrame(summary_rows)
summary_df


,class_for_check,file,real_available,synthetic_available,real_sampled,synthetic_sampled,total_sampled,status
0,Проблема с качеством товара,labeled/wb_feedbacks_by_class_with_synthetic/п...,528,0,10,0,10,ok
1,Проблема с размером / посадкой,labeled/wb_feedbacks_by_class_with_synthetic/п...,141,59,10,10,20,ok
2,Несоответствие карточке товара,labeled/wb_feedbacks_by_class_with_synthetic/н...,210,0,10,0,10,ok
3,Проблема с комплектацией / упаковкой,labeled/wb_feedbacks_by_class_with_synthetic/п...,392,0,10,0,10,ok
4,Проблема доставки / получения,labeled/wb_feedbacks_by_class_with_synthetic/п...,82,118,10,10,20,ok
5,Цена / ценность,labeled/wb_feedbacks_by_class_with_synthetic/ц...,48,152,10,10,20,ok
6,Проблема с возвратом,labeled/wb_feedbacks_by_class_with_synthetic/п...,238,0,10,0,10,ok
7,Положительный / нейтральный отзыв,labeled/wb_feedbacks_by_class_with_synthetic/п...,85,115,10,10,20,ok
8,Другая проблема,labeled/wb_feedbacks_by_class_with_synthetic/д...,20,180,10,10,20,ok


## 5. Сохранение итогового файла для ручной проверки

In [7]:
if not sample_parts:
    raise RuntimeError("Не удалось сформировать выборку: нет подходящих строк.")

check_df = pd.concat(sample_parts, ignore_index=True)

# Перемешиваем итоговую выборку, чтобы при ручной проверке классы не шли блоками.
check_df = check_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

# Создаем колонки для ручной проверки.
check_df["is_correct"] = ""
check_df["correct_labels"] = ""
check_df["comment"] = ""

# Оставляем удобный набор колонок.
preferred_cols = [
    "manual_check_id",
    "class_for_check",
    "source_type",
    TEXT_COLUMN,
    LABELS_COLUMN,
    "labels_str",
    "is_correct",
    "correct_labels",
    "comment",
    "source_class_file",
]

existing_cols = [col for col in preferred_cols if col in check_df.columns]
check_df = check_df[existing_cols]

check_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
summary_df.to_csv(SUMMARY_OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("Итоговый файл для ручной проверки сохранен:")
print(OUTPUT_PATH)
print()
print("Сводка сохранена:")
print(SUMMARY_OUTPUT_PATH)
print()
print("Размер выборки:", len(check_df))

check_df.head(20)


Итоговый файл для ручной проверки сохранен:
/content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_manual_check_random/manual_check_random_by_class.csv

Сводка сохранена:
/content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_manual_check_random/manual_check_random_summary.csv

Размер выборки: 140


,manual_check_id,class_for_check,source_type,отзыв,labels,labels_str,is_correct,correct_labels,comment,source_class_file
0,положительный_нейтральный_отзыв__8,Положительный / нейтральный отзыв,real,"Мне очень понравилось платье, лично такая ткан...","[""Положительный / нейтральный отзыв""]",Положительный / нейтральный отзыв,,,,labeled/wb_feedbacks_by_class_with_synthetic/п...
1,проблема_доставки_получения__17,Проблема доставки / получения,synthetic,"Курьер позвонил и сказал, что не может достави...","[""Проблема доставки / получения""]",Проблема доставки / получения,,,,labeled/wb_feedbacks_by_class_with_synthetic/п...
2,несоответствие_карточке_товара__1,Несоответствие карточке товара,real,Купила куклу Алису на новый год ребенку. Обман...,"[""Несоответствие карточке товара""]",Несоответствие карточке товара,,,,labeled/wb_feedbacks_by_class_with_synthetic/н...
3,положительный_нейтральный_отзыв__19,Положительный / нейтральный отзыв,synthetic,"Ремешок подошёл к часам, все как в описании, д...","[""Положительный / нейтральный отзыв""]",Положительный / нейтральный отзыв,,,,labeled/wb_feedbacks_by_class_with_synthetic/п...
4,проблема_с_комплектацией_упаковкой__2,Проблема с комплектацией / упаковкой,real,"Пришёл открытый, учитывая специфику товара, сч...","[""Проблема с комплектацией / упаковкой"", ""Проб...",Проблема с комплектацией / упаковкой | Проблем...,,,,labeled/wb_feedbacks_by_class_with_synthetic/п...
5,проблема_с_размером_посадкой__2,Проблема с размером / посадкой,real,"Сапожки красивые, на ножке смотрятся великолеп...","[""Проблема с размером / посадкой""]",Проблема с размером / посадкой,,,,labeled/wb_feedbacks_by_class_with_synthetic/п...
6,цена_ценность__11,Цена / ценность,synthetic,"Детский набор совсем маленький, объем игрушек ...","[""Цена / ценность""]",Цена / ценность,,,,labeled/wb_feedbacks_by_class_with_synthetic/ц...
7,проблема_доставки_получения__19,Проблема доставки / получения,synthetic,"Очень неудобно сделали, что теперь выдача толь...","[""Проблема доставки / получения""]",Проблема доставки / получения,,,,labeled/wb_feedbacks_by_class_with_synthetic/п...
8,положительный_нейтральный_отзыв__4,Положительный / нейтральный отзыв,real,"Очень полезная и качественная вещь, все пришло...","[""Положительный / нейтральный отзыв""]",Положительный / нейтральный отзыв,,,,labeled/wb_feedbacks_by_class_with_synthetic/п...
9,положительный_нейтральный_отзыв__9,Положительный / нейтральный отзыв,real,Товар пришел в упаковке без брака.Фольгу и пле...,"[""Положительный / нейтральный отзыв""]",Положительный / нейтральный отзыв,,,,labeled/wb_feedbacks_by_class_with_synthetic/п...


## 6. Быстрая проверка, сколько примеров попало по каждому классу и source_type

In [8]:
quality_sample_stats = (
    check_df
    .groupby(["class_for_check", "source_type"])
    .size()
    .reset_index(name="sample_count")
    .sort_values(["class_for_check", "source_type"])
)

quality_sample_stats


,class_for_check,source_type,sample_count
0,Другая проблема,real,10
1,Другая проблема,synthetic,10
2,Несоответствие карточке товара,real,10
3,Положительный / нейтральный отзыв,real,10
4,Положительный / нейтральный отзыв,synthetic,10
5,Проблема доставки / получения,real,10
6,Проблема доставки / получения,synthetic,10
7,Проблема с возвратом,real,10
8,Проблема с качеством товара,real,10
9,Проблема с комплектацией / упаковкой,real,10


## 7. Как заполнять файл вручную

В итоговом CSV заполни:

```text
is_correct = 1, если labels подходят отзыву
is_correct = 0, если labels ошибочные
correct_labels = правильные labels, если is_correct = 0
comment = короткий комментарий, если нужно
```

Пример:

```text
is_correct: 0
correct_labels: ["Несоответствие карточке товара"]
comment: модель поставила качество, но в отзыве сравнение с фото
```


## 8. Опционально: подсчет качества после ручной проверки

Запусти эту ячейку после того, как заполнишь `is_correct` в CSV.


In [9]:
CHECKED_PATH = OUTPUT_PATH

checked_df = read_csv_safely(CHECKED_PATH)

checked_df["is_correct_num"] = pd.to_numeric(
    checked_df["is_correct"],
    errors="coerce",
)

checked_only = checked_df.dropna(subset=["is_correct_num"]).copy()

if checked_only.empty:
    print("Пока нет заполненных строк is_correct.")
else:
    quality_by_class = (
        checked_only
        .groupby(["class_for_check", "source_type"])["is_correct_num"]
        .agg(["count", "mean"])
        .reset_index()
    )

    quality_by_class["accuracy_percent"] = (quality_by_class["mean"] * 100).round(1)

    display(quality_by_class.sort_values(["accuracy_percent", "count"]))


Пока нет заполненных строк is_correct.


In [11]:
# ============================================================
# ПЕРЕСБОРКА ФАЙЛА ДЛЯ РУЧНОЙ ПРОВЕРКИ НА ТЕХ ЖЕ САМЫХ ПРИМЕРАХ
# ============================================================

from pathlib import Path
import shutil
import json
import re
from collections import defaultdict

import pandas as pd


# ------------------------------------------------------------
# 1. ПУТИ
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/MLops_project/project/data").resolve()
LABELED_DIR = PROJECT_ROOT / "labeled"

# Новая папка с файлами по классам, полученная после нового ChatGPT_markup
NEW_BY_CLASS_DIR = LABELED_DIR / "wb_feedbacks_by_class_with_synthetic"

# Старый файл ручной проверки, где были те самые примеры
# ВАЖНО: если старый файл лежит в другом месте, поменяй путь здесь
OLD_MANUAL_CHECK_PATH = (
    LABELED_DIR
    / "wb_feedbacks_manual_check_random_v1"
    / "manual_check_random_by_class.csv"
)

# Куда перезаписываем новую ручную проверку
OUTPUT_DIR = LABELED_DIR / "wb_feedbacks_manual_check_random"
OUTPUT_PATH = OUTPUT_DIR / "manual_check_random_by_class.csv"
SUMMARY_OUTPUT_PATH = OUTPUT_DIR / "manual_check_random_summary.csv"


# ------------------------------------------------------------
# 2. НАСТРОЙКИ
# ------------------------------------------------------------

TEXT_COLUMN = "отзыв"
LABELS_COLUMN = "labels"

ALLOWED_LABELS_MVP = [
    "Проблема с качеством товара",
    "Проблема с размером / посадкой",
    "Несоответствие карточке товара",
    "Проблема с комплектацией / упаковкой",
    "Проблема доставки / получения",
    "Цена / ценность",
    "Проблема с возвратом",
    "Положительный / нейтральный отзыв",
    "Другая проблема",
]

ALLOWED_LABELS_SET = set(ALLOWED_LABELS_MVP)


# ------------------------------------------------------------
# 3. ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ------------------------------------------------------------

def read_csv_safely(path: Path) -> pd.DataFrame:
    encodings = ["utf-8", "utf-8-sig", "cp1251"]
    last_error = None

    for encoding in encodings:
        try:
            return pd.read_csv(path, encoding=encoding)
        except Exception as e:
            last_error = e

    raise RuntimeError(f"Не удалось прочитать файл {path}. Последняя ошибка: {last_error}")


def safe_filename(label: str) -> str:
    name = label.lower()
    name = name.replace(" / ", "_")
    name = name.replace("/", "_")
    name = name.replace(" ", "_")
    name = re.sub(r"[^а-яa-z0-9_]+", "", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name + ".csv"


def labels_to_list(labels):
    if isinstance(labels, list):
        raw = labels
    elif pd.isna(labels):
        raw = []
    elif isinstance(labels, str):
        value = labels.strip()

        if not value:
            raw = []
        else:
            try:
                parsed = json.loads(value)
                raw = parsed if isinstance(parsed, list) else []
            except Exception:
                if " | " in value:
                    raw = [x.strip() for x in value.split(" | ")]
                elif value in ALLOWED_LABELS_SET:
                    raw = [value]
                else:
                    raw = []
    else:
        raw = []

    result = []
    for label in raw:
        label = str(label).strip()
        if label in ALLOWED_LABELS_SET and label not in result:
            result.append(label)

    return result


def normalize_labels(labels):
    result = labels_to_list(labels)

    if not result:
        result = ["Другая проблема"]

    neutral_label = "Положительный / нейтральный отзыв"

    if neutral_label in result and len(result) > 1:
        result = [label for label in result if label != neutral_label]

    if not result:
        result = ["Другая проблема"]

    return result


def normalize_text(text):
    return str(text).strip()


# ------------------------------------------------------------
# 4. ПРОВЕРКИ
# ------------------------------------------------------------

if not OLD_MANUAL_CHECK_PATH.exists():
    raise FileNotFoundError(
        f"Не найден старый файл ручной проверки: {OLD_MANUAL_CHECK_PATH}\n"
        f"Поменяй OLD_MANUAL_CHECK_PATH на реальный путь к старому файлу."
    )

if not NEW_BY_CLASS_DIR.exists():
    raise FileNotFoundError(
        f"Не найдена новая папка с файлами по классам: {NEW_BY_CLASS_DIR}"
    )


# ------------------------------------------------------------
# 5. ЧИТАЕМ СТАРУЮ ВЫБОРКУ
# ------------------------------------------------------------

old_check_df = read_csv_safely(OLD_MANUAL_CHECK_PATH)

if TEXT_COLUMN not in old_check_df.columns:
    raise ValueError(
        f"В старом файле нет колонки {TEXT_COLUMN!r}. "
        f"Доступные колонки: {list(old_check_df.columns)}"
    )

old_check_df = old_check_df.copy()
old_check_df[TEXT_COLUMN] = old_check_df[TEXT_COLUMN].apply(normalize_text)

print("Старый файл ручной проверки:")
print(OLD_MANUAL_CHECK_PATH)
print("Строк:", len(old_check_df))


# ------------------------------------------------------------
# 6. ЧИТАЕМ НОВЫЕ ФАЙЛЫ ПО КЛАССАМ И СОБИРАЕМ НОВЫЕ LABELS ПО ТЕКСТУ ОТЗЫВА
# ------------------------------------------------------------

labels_by_text = defaultdict(list)
source_type_by_text = {}
source_files_by_text = defaultdict(list)

summary_rows = []

for label in ALLOWED_LABELS_MVP:
    class_path = NEW_BY_CLASS_DIR / safe_filename(label)

    if not class_path.exists():
        print("Файл класса не найден, пропускаю:", class_path)
        summary_rows.append({
            "class_for_check": label,
            "file": str(class_path),
            "rows": 0,
            "status": "file_not_found",
        })
        continue

    part = read_csv_safely(class_path)

    if TEXT_COLUMN not in part.columns:
        raise ValueError(
            f"В файле {class_path} нет колонки {TEXT_COLUMN!r}. "
            f"Доступные колонки: {list(part.columns)}"
        )

    if LABELS_COLUMN not in part.columns:
        raise ValueError(
            f"В файле {class_path} нет колонки {LABELS_COLUMN!r}. "
            f"Доступные колонки: {list(part.columns)}"
        )

    part = part.copy()
    part[TEXT_COLUMN] = part[TEXT_COLUMN].fillna("").astype(str).str.strip()
    part = part[part[TEXT_COLUMN].ne("")].copy()

    if "source_type" not in part.columns:
        part["source_type"] = "real"

    part["source_type"] = (
        part["source_type"]
        .fillna("real")
        .astype(str)
        .str.lower()
        .str.strip()
    )

    part.loc[
        part["source_type"].isin(["synth", "synthetic", "generated"]),
        "source_type"
    ] = "synthetic"

    part.loc[
        ~part["source_type"].isin(["real", "synthetic"]),
        "source_type"
    ] = "real"

    part["labels_list"] = part[LABELS_COLUMN].apply(normalize_labels)

    for _, row in part.iterrows():
        text = normalize_text(row[TEXT_COLUMN])

        # Берем labels из строки
        for lab in row["labels_list"]:
            if lab not in labels_by_text[text]:
                labels_by_text[text].append(lab)

        # Дополнительно добавляем сам класс файла, если он реально является разрешенным label
        if label in ALLOWED_LABELS_SET and label not in labels_by_text[text]:
            labels_by_text[text].append(label)

        source_type_by_text[text] = row["source_type"]
        source_files_by_text[text].append(str(class_path.relative_to(PROJECT_ROOT)))

    summary_rows.append({
        "class_for_check": label,
        "file": str(class_path.relative_to(PROJECT_ROOT)),
        "rows": len(part),
        "status": "ok",
    })

print()
print("Уникальных отзывов в новой папке по классам:", len(labels_by_text))


# ------------------------------------------------------------
# 7. ОБНОВЛЯЕМ LABELS В СТАРОЙ ВЫБОРКЕ, СОХРАНЯЯ ТЕ ЖЕ САМЫЕ ПРИМЕРЫ И ПОРЯДОК
# ------------------------------------------------------------

new_check_df = old_check_df.copy()

new_labels = []
new_labels_str = []
new_source_type = []
new_source_class_file = []
match_status = []

for _, row in new_check_df.iterrows():
    text = normalize_text(row[TEXT_COLUMN])

    if text in labels_by_text:
        labels_list = labels_by_text[text]
        status = "found_in_new_by_class"

        new_labels.append(json.dumps(labels_list, ensure_ascii=False))
        new_labels_str.append(" | ".join(labels_list))

        new_source_type.append(
            source_type_by_text.get(text, row.get("source_type", "real"))
        )

        source_files = source_files_by_text.get(text, [])
        new_source_class_file.append(" ; ".join(sorted(set(source_files))))

    else:
        # Если отзыв не найден в новой папке, оставляем старые labels,
        # чтобы строка не потерялась.
        old_labels = normalize_labels(row.get(LABELS_COLUMN, ""))
        status = "not_found_in_new_by_class_keep_old_labels"

        new_labels.append(json.dumps(old_labels, ensure_ascii=False))
        new_labels_str.append(" | ".join(old_labels))

        new_source_type.append(row.get("source_type", ""))
        new_source_class_file.append(row.get("source_class_file", ""))

    match_status.append(status)

new_check_df[LABELS_COLUMN] = new_labels
new_check_df["labels_str"] = new_labels_str
new_check_df["source_type"] = new_source_type
new_check_df["source_class_file"] = new_source_class_file
new_check_df["match_status"] = match_status


# ------------------------------------------------------------
# 8. ОЧИЩАЕМ КОЛОНКИ РУЧНОЙ ПРОВЕРКИ
# ------------------------------------------------------------

new_check_df["is_correct"] = ""
new_check_df["correct_labels"] = ""
new_check_df["comment"] = ""


# ------------------------------------------------------------
# 9. ОСТАВЛЯЕМ УДОБНЫЙ НАБОР КОЛОНОК
# ------------------------------------------------------------

preferred_cols = [
    "manual_check_id",
    "class_for_check",
    "source_type",
    TEXT_COLUMN,
    LABELS_COLUMN,
    "labels_str",
    "is_correct",
    "correct_labels",
    "comment",
    "source_class_file",
    "match_status",
]

existing_cols = [col for col in preferred_cols if col in new_check_df.columns]
new_check_df = new_check_df[existing_cols]


# ------------------------------------------------------------
# 10. ПЕРЕЗАПИСЫВАЕМ ПАПКУ ДЛЯ РУЧНОЙ ПРОВЕРКИ
# ------------------------------------------------------------

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

new_check_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

summary_df = pd.DataFrame(summary_rows)

result_summary = pd.DataFrame([
    {
        "metric": "old_manual_check_rows",
        "value": len(old_check_df),
    },
    {
        "metric": "new_manual_check_rows",
        "value": len(new_check_df),
    },
    {
        "metric": "found_in_new_by_class",
        "value": int((new_check_df["match_status"] == "found_in_new_by_class").sum()),
    },
    {
        "metric": "not_found_in_new_by_class_keep_old_labels",
        "value": int((new_check_df["match_status"] == "not_found_in_new_by_class_keep_old_labels").sum()),
    },
])

with pd.ExcelWriter(OUTPUT_DIR / "manual_check_rebuild_report.xlsx") as writer:
    summary_df.to_excel(writer, index=False, sheet_name="class_files_summary")
    result_summary.to_excel(writer, index=False, sheet_name="result_summary")

summary_df.to_csv(SUMMARY_OUTPUT_PATH, index=False, encoding="utf-8-sig")

print()
print("=" * 100)
print("Готово.")
print("Папка ручной проверки перезаписана:")
print(OUTPUT_DIR)
print()
print("Новый файл ручной проверки:")
print(OUTPUT_PATH)
print()
print("Строк в старом файле:", len(old_check_df))
print("Строк в новом файле:", len(new_check_df))
print()
print("Найдено тех же отзывов в новой папке по классам:")
print((new_check_df["match_status"] == "found_in_new_by_class").sum())
print()
print("Не найдено, labels оставлены старыми:")
print((new_check_df["match_status"] == "not_found_in_new_by_class_keep_old_labels").sum())
print("=" * 100)

new_check_df.head(20)

Старый файл ручной проверки:
/content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_manual_check_random_v1/manual_check_random_by_class.csv
Строк: 150
Файл класса не найден, пропускаю: /content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_by_class_with_synthetic/подделка_оригинальность.csv

Уникальных отзывов в новой папке по классам: 1805

Готово.
Папка ручной проверки перезаписана:
/content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_manual_check_random

Новый файл ручной проверки:
/content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_manual_check_random/manual_check_random_by_class.csv

Строк в старом файле: 150
Строк в новом файле: 150

Найдено тех же отзывов в новой папке по классам:
88

Не найдено, labels оставлены старыми:
62


,manual_check_id,class_for_check,source_type,отзыв,labels,labels_str,is_correct,correct_labels,comment,source_class_file,match_status
0,проблема_доставки_получения__13,Проблема доставки / получения,synthetic,"Перенесли доставку на неделю, и к тому же това...","[""Проблема доставки / получения"", ""Проблема с ...",Проблема доставки / получения | Проблема с ком...,,,,labeled/wb_feedbacks_by_class_with_synthetic/п...,not_found_in_new_by_class_keep_old_labels
1,проблема_с_размером_посадкой__8,Проблема с размером / посадкой,real,"Свитер большемерит. На рост ребенка 116, р-о 1...","[""Проблема с размером / посадкой"", ""Проблема с...",Проблема с размером / посадкой | Проблема с ка...,,,,labeled/wb_feedbacks_by_class_with_synthetic/п...,found_in_new_by_class
2,положительный_нейтральный_отзыв__8,Положительный / нейтральный отзыв,real,"Отличная куртка, размер в размер. Дочь довольн...","[""Положительный / нейтральный отзыв""]",Положительный / нейтральный отзыв,,,,labeled/wb_feedbacks_by_class_with_synthetic/п...,found_in_new_by_class
3,проблема_доставки_получения__18,Проблема доставки / получения,synthetic,"Доставка заняла больше месяца, ожидал намного ...","[""Проблема доставки / получения""]",Проблема доставки / получения,,,,labeled/wb_feedbacks_by_class_with_synthetic/п...,not_found_in_new_by_class_keep_old_labels
4,проблема_доставки_получения__16,Проблема доставки / получения,synthetic,"На пункте выдачи не было моего заказа, сказали...","[""Проблема доставки / получения""]",Проблема доставки / получения,,,,labeled/wb_feedbacks_by_class_with_synthetic/п...,not_found_in_new_by_class_keep_old_labels
5,несоответствие_карточке_товара__1,Несоответствие карточке товара,real,"Прислали совсем не crocs а другую фирму, с веш...","[""Несоответствие карточке товара""]",Несоответствие карточке товара,,,,labeled/wb_feedbacks_by_class_with_synthetic/н...,found_in_new_by_class
6,проблема_доставки_получения__4,Проблема доставки / получения,real,"Пришло не то, ладно бы был блич, но это даже б...","[""Несоответствие карточке товара"", ""Проблема д...",Несоответствие карточке товара | Проблема дост...,,,,labeled/wb_feedbacks_by_class_with_synthetic/н...,found_in_new_by_class
7,другая_проблема__11,Другая проблема,synthetic,"Инструкция вообще непонятная, не хватает карти...","[""Другая проблема""]",Другая проблема,,,,labeled/wb_feedbacks_by_class_with_synthetic/д...,not_found_in_new_by_class_keep_old_labels
8,проблема_доставки_получения__8,Проблема доставки / получения,real,Сам чемодан супер!!! Но доставка 🤦🏼‍♀️ коробка...,"[""Проблема с комплектацией / упаковкой""]",Проблема с комплектацией / упаковкой,,,,labeled/wb_feedbacks_by_class_with_synthetic/п...,found_in_new_by_class
9,цена_ценность__2,Цена / ценность,real,Открывашка отстой! Враньё и ещё раз враньё! Эт...,"[""Проблема с качеством товара"", ""Несоответстви...",Проблема с качеством товара | Несоответствие к...,,,,labeled/wb_feedbacks_by_class_with_synthetic/н...,found_in_new_by_class
